# 面试问题：Tree of Thoughts 怎样在固定预算下生成、评估、剪枝并回溯，而不是只走贪心 CoT？

        ## 可直接复述的回答主线

        1. Tree of Thoughts 把中间思路当作可扩展状态，在多个候选之间搜索，而普通 CoT 通常只保留一条路径。
2. 朴素贪心会被高即时分的表面修复吸引，丢掉需要先诊断再处理的低即时分路径。
3. 底层循环包含 expand、score、beam prune、预算计数和终态判断，每一步都应写入搜索账本。
4. 同一案例要比较贪心路径、beam 搜索路径、累计分数、展开节点数和是否命中正确修复。
5. beam 太窄或预算小于所需深度时仍会失败，因此预算本身是算法参数。
6. 生产 Agent 还必须将模拟思路与真实工具执行分离，并对执行动作重新鉴权。

        后续实验会用同一批输入依次验证朴素方案、核心机制、失败边界和修正效果。

## 1. 真实案例与输入预览

案例是五条 LLM Gateway 运维事件，字段包括症状、表面修复、诊断动作、真正修复和错误修复。每条事件的表面修复即时分较高但后续无效，诊断路径需要两步才能获得更高总分，用于真实复现贪心搜索失败。

In [1]:
incidents = [{"id": "inc-01", "symptom": "首Token延迟突增且GPU利用率低", "shortcut": "重启网关", "diagnosis": "检查调度队列", "fix": "降低最大批大小", "bad_fix": "增加客户端超时"}, {"id": "inc-02", "symptom": "单租户大量429", "shortcut": "关闭限流", "diagnosis": "检查租户配额", "fix": "修正该租户配额", "bad_fix": "扩大所有租户配额"}, {"id": "inc-03", "symptom": "长提示请求频繁OOM", "shortcut": "重试请求", "diagnosis": "检查KV块占用", "fix": "限制长上下文准入", "bad_fix": "提高重试次数"}, {"id": "inc-04", "symptom": "流式响应重复Token", "shortcut": "刷新页面", "diagnosis": "检查重连游标", "fix": "按事件ID去重续传", "bad_fix": "关闭流式输出"}, {"id": "inc-05", "symptom": "工具调用偶发重复扣款", "shortcut": "重启Agent", "diagnosis": "检查幂等键", "fix": "提交前绑定幂等键", "bad_fix": "增加模型温度"}]  # 定义五条具有诊断和修复语义的运维事件。
print("教学实验输入：五条 LLM Gateway 事件")  # 标记下方为离线事件样本。
print("事件      症状                         表面修复       诊断动作       正确修复")  # 输出事件字段表头。
for incident in incidents:  # 逐条展示症状与候选行动。
    print(f"{incident['id']:<9} {incident['symptom']:<28} {incident['shortcut']:<12} {incident['diagnosis']:<12} {incident['fix']}")  # 输出当前运维案例。

教学实验输入：五条 LLM Gateway 事件
事件      症状                         表面修复       诊断动作       正确修复
inc-01    首Token延迟突增且GPU利用率低           重启网关         检查调度队列       降低最大批大小
inc-02    单租户大量429                     关闭限流         检查租户配额       修正该租户配额
inc-03    长提示请求频繁OOM                   重试请求         检查KV块占用      限制长上下文准入
inc-04    流式响应重复Token                  刷新页面         检查重连游标       按事件ID去重续传
inc-05    工具调用偶发重复扣款                   重启Agent      检查幂等键        提交前绑定幂等键


## 2. Baseline / 基线：每一步只选即时分最高的 Thought

根节点给表面修复 0.82 分，给诊断动作 0.48 分。贪心会立即保留表面修复，第二步发现问题未改善后也无法回到已经丢弃的诊断分支。

In [2]:
def expand(incident, path):  # 根据当前思路路径生成下一步候选和局部分数。
    if len(path) == 0:  # 根节点同时生成表面修复和诊断分支。
        return [(incident["shortcut"], 0.82), (incident["diagnosis"], 0.48)]  # 返回两个具有不同即时吸引力的候选。
    if path[-1] == incident["shortcut"]:  # 表面修复后只能观察到问题未改善。
        return [("观察:问题未改善", -0.25)]  # 返回负反馈终止分支。
    if path[-1] == incident["diagnosis"]:  # 诊断动作后生成正确和错误修复。
        return [(incident["fix"], 1.05), (incident["bad_fix"], 0.10)]  # 返回两个可比较修复候选。
    return []  # 两步之后视为终态不再扩展。
def greedy_search(incident, depth):  # 实现只保留最高即时分的朴素推理。
    path = []  # 初始化空思路路径。
    total_score = 0.0  # 初始化累计评估分。
    for _ in range(depth):  # 在固定深度内逐步生成。
        candidates = expand(incident, path)  # 生成当前路径的下一步候选。
        if not candidates:  # 终态没有候选时提前结束。
            break  # 停止继续扩展当前路径。
        action, local_score = max(candidates, key=lambda item: (item[1], item[0]))  # 仅按即时分选择一步。
        path.append(action)  # 把贪心动作加入路径。
        total_score += local_score  # 累加当前局部分数。
    return tuple(path), total_score  # 返回贪心路径和累计分。
baseline_results = [{"id": incident["id"], "path": greedy_search(incident, 2)[0], "score": greedy_search(incident, 2)[1], "success": greedy_search(incident, 2)[0] == (incident["diagnosis"], incident["fix"])} for incident in incidents]  # 对五条事件执行两步贪心。
print("Baseline 贪心路径")  # 标记当前输出只保留一条 Thought。
for result in baseline_results:  # 逐事件展示贪心路径和结果。
    print(f"{result['id']} path={result['path']} score={result['score']:.2f} success={result['success']}")  # 输出当前事件的贪心失败。

Baseline 贪心路径
inc-01 path=('重启网关', '观察:问题未改善') score=0.57 success=False
inc-02 path=('关闭限流', '观察:问题未改善') score=0.57 success=False
inc-03 path=('重试请求', '观察:问题未改善') score=0.57 success=False
inc-04 path=('刷新页面', '观察:问题未改善') score=0.57 success=False
inc-05 path=('重启Agent', '观察:问题未改善') score=0.57 success=False


## 3. 底层实现：预算化 Beam Tree Search

frontier 保存路径和累计分。每层展开所有保留路径，把每个候选写入 ledger，再按累计分剪到 beam width；预算按实际生成的 child 数扣减。

In [3]:
def beam_search(incident, depth, beam_width, budget):  # 在固定深度、beam 和节点预算下搜索思路树。
    frontier = [(tuple(), 0.0)]  # 用空路径和零分初始化搜索前沿。
    ledger = []  # 保存每个展开节点的深度、路径、局部分和累计分。
    expanded = 0  # 记录已经消耗的候选生成预算。
    for level in range(depth):  # 按层执行生成、评估和剪枝。
        candidates = []  # 收集当前层所有可用 child。
        for path, score in frontier:  # 遍历上一层保留的思路。
            for action, local_score in expand(incident, list(path)):  # 扩展当前思路的下一步。
                if expanded >= budget:  # 到达节点预算时停止新增候选。
                    break  # 跳出当前路径的候选生成。
                new_path = path + (action,)  # 构造不可变新路径以便稳定记录。
                new_score = score + local_score  # 累加父路径和当前局部分。
                candidates.append((new_path, new_score))  # 把新 Thought 加入本层候选。
                ledger.append({"level": level + 1, "path": new_path, "local": local_score, "total": new_score})  # 保存搜索过程供教学观察。
                expanded += 1  # 扣减一个节点生成预算。
        if not candidates:  # 没有新候选说明搜索已终止或预算耗尽。
            break  # 停止后续层扩展。
        frontier = sorted(candidates, key=lambda item: (-item[1], item[0]))[:beam_width]  # 按累计分保留固定宽度的最佳路径。
    best_path, best_score = max(frontier, key=lambda item: (item[1], item[0]))  # 从最终前沿选择累计分最高路径。
    return best_path, best_score, ledger, expanded  # 返回最佳路径、分数、账本和预算消耗。
first_path, first_score, first_ledger, first_expanded = beam_search(incidents[0], depth=2, beam_width=2, budget=8)  # 对第一条事件执行可回溯 beam 搜索。
print("inc-01 Tree of Thoughts 搜索账本")  # 标记下表展示生成、评估和剪枝前的所有 child。
print("层  local  total  path")  # 输出搜索账本表头。
for item in first_ledger:  # 逐节点展示搜索过程。
    print(f"{item['level']:>2} {item['local']:>6.2f} {item['total']:>6.2f}  {item['path']}")  # 输出当前 Thought 的分项得分。
print(f"最终保留={first_path}，score={first_score:.2f}，expanded={first_expanded}")  # 展示搜索选择与预算消耗。

inc-01 Tree of Thoughts 搜索账本
层  local  total  path
 1   0.82   0.82  ('重启网关',)
 1   0.48   0.48  ('检查调度队列',)
 2  -0.25   0.57  ('重启网关', '观察:问题未改善')
 2   1.05   1.53  ('检查调度队列', '降低最大批大小')
 2   0.10   0.58  ('检查调度队列', '增加客户端超时')
最终保留=('检查调度队列', '降低最大批大小')，score=1.53，expanded=5


## 4. 结果表与结果解读

同样深度二，beam width 二保留低即时分的诊断分支，使它在第二步凭正确修复反超。五条事件都使用同一搜索规则，不针对单题改参数。

In [4]:
search_results = []  # 保存五条事件的 ToT 最佳路径和预算。
for incident in incidents:  # 对同一批事件运行相同 beam 和预算。
    path, score, ledger, expanded = beam_search(incident, depth=2, beam_width=2, budget=8)  # 执行两层宽度二搜索。
    search_results.append({"id": incident["id"], "path": path, "score": score, "expanded": expanded, "success": path == (incident["diagnosis"], incident["fix"])})  # 保存是否命中正确诊断修复。
baseline_success = sum(result["success"] for result in baseline_results)  # 统计贪心成功事件数。
search_success = sum(result["success"] for result in search_results)  # 统计 ToT 成功事件数。
print("事件      贪心score  ToT score  展开节点  ToT路径正确")  # 输出同事件、同深度结果表头。
for baseline_result, search_result in zip(baseline_results, search_results):  # 逐事件比较贪心和 ToT。
    print(f"{search_result['id']:<9} {baseline_result['score']:>9.2f} {search_result['score']:>9.2f} {search_result['expanded']:>9} {str(search_result['success']):>11}")  # 输出当前事件的搜索对照。
print(f"结果解读：贪心成功={baseline_success}/{len(incidents)}，ToT成功={search_success}/{len(incidents)}；提升来自保留诊断分支，不代表真实模型评分可靠。")  # 解释收益来源和外推边界。

事件      贪心score  ToT score  展开节点  ToT路径正确
inc-01         0.57      1.53         5        True
inc-02         0.57      1.53         5        True
inc-03         0.57      1.53         5        True
inc-04         0.57      1.53         5        True
inc-05         0.57      1.53         5        True
结果解读：贪心成功=0/5，ToT成功=5/5；提升来自保留诊断分支，不代表真实模型评分可靠。


## 5. 失败案例与修正

beam width 为一时，算法退化为按累计分保留单一路径，第一层仍丢掉诊断。修正为 width 二并给足至少两层预算；真实系统应按边际收益动态分配。

In [5]:
narrow_path, narrow_score, narrow_ledger, narrow_expanded = beam_search(incidents[0], depth=2, beam_width=1, budget=8)  # 故意把 beam 缩到一复现搜索退化。
fixed_path, fixed_score, fixed_ledger, fixed_expanded = beam_search(incidents[0], depth=2, beam_width=2, budget=8)  # 使用宽度二恢复被贪心丢弃的诊断分支。
print(f"错误行为：beam=1 path={narrow_path} score={narrow_score:.2f} success={narrow_path == (incidents[0]['diagnosis'], incidents[0]['fix'])}")  # 展示窄 beam 的失败路径。
print(f"修正行为：beam=2 path={fixed_path} score={fixed_score:.2f} success={fixed_path == (incidents[0]['diagnosis'], incidents[0]['fix'])}")  # 展示扩大搜索宽度后的正确路径。

错误行为：beam=1 path=('重启网关', '观察:问题未改善') score=0.57 success=False
修正行为：beam=2 path=('检查调度队列', '降低最大批大小') score=1.53 success=True


## 6. 生产边界

这里的分数是离线规则，不是 LLM 自评可靠性。生产 Agent 需要区分 Thought simulation 与工具 commit，给工具调用设置审批、幂等键、超时和权威回读，并监控搜索成本与收益。

In [6]:
total_expanded = sum(result["expanded"] for result in search_results)  # 汇总五条事件实际生成的 Thought 数。
production_metrics = {"incidents": len(incidents), "expanded_nodes": total_expanded, "successes": search_success, "tool_calls_committed": 0}  # 明确本实验只搜索没有真实执行工具。
print("生产边界快照：", production_metrics)  # 输出搜索成本与零真实副作用。

生产边界快照： {'incidents': 5, 'expanded_nodes': 25, 'successes': 5, 'tool_calls_committed': 0}


## 7. 最小回归测试

只验证案例规模、预算上限、贪心失败和宽度修正。

In [7]:
assert len(incidents) >= 5  # 保证案例至少包含五条真实字段运维事件。
assert all(result["expanded"] <= 8 for result in search_results)  # 保证每条搜索都遵守节点预算。
assert baseline_success == 0  # 保证教学输入真实复现高即时分贪心失败。
assert search_success == len(incidents)  # 保证宽度二搜索在同一规则下找到五条正确修复。
assert narrow_path != fixed_path  # 保证失败修正实际改变搜索路径。